In [8]:
import os
import sys
project_path = os.path.abspath(os.path.join(os.getcwd(),'..'))
sys.path.insert(0,project_path)

#Config and connection testing
from google.cloud import bigquery
from src.config import settings

# For EDA, Data prepareation, Data visulisation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#For network plot
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations
from collections import Counter

from datetime import datetime, timedelta
import logging
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


from src.data_explorer import DataAnalyzer
from src.data_argumenter import OrderAwareAugmentation
from src.data_feature_engineering import OrderLevelFeatureEngineering
from src.data_prepareation import DataPreparation
from src.baseline import RecommenderFactory,ModelType


In [2]:
# #Create dataset
# clinet = bigquery.Client(project=settings.project_id)
# df = clinet.query("""
# SELECT 
#     order_id,
#     order_item_id,
#     order_date,
#     order_time,
#     round(order_total_price / 100,1) as order_total_price,
#     round(order_gst / 100,2) as order_gst,
#     cart_order_time,
# product_name,
# product_variant,
# product_category,
# customer_id,
# first_order_date,
# last_order_date,
# total_orders,
# total_lifetime_value,
# customer_segment
# FROM dbt_medallion_dev_gold.fact_order_items src
# left join dbt_medallion_dev_gold.dim_products dp on src.product_key = dp.product_key
# left join dbt_medallion_dev_gold.dim_customers dc on src.customer_key = dc.customer_key
# order by order_id

# """).to_dataframe()

#Import base df
df = pd.read_csv('../data/raw/data_raw.csv',index_col=0)

In [ ]:
# analyzer = DataAnalyzer(df)
# results = analyzer.run_analysis()

In [3]:
prep = DataPreparation(df)
clean_df, report = prep.run_preparation()

2025-11-24 21:54:01,784 - src.data_prepareation - INFO - Removed 0 duplicate rows
2025-11-24 21:54:01,796 - src.data_prepareation - INFO - Columns with missing values:
2025-11-24 21:54:01,796 - src.data_prepareation - INFO -   product_variant: 1061 (1.5%)
2025-11-24 21:54:01,798 - src.data_prepareation - INFO - Filled missing product_variant with 'Regular'
2025-11-24 21:54:01,818 - src.data_prepareation - INFO - All product categories are consistent
2025-11-24 21:54:01,819 - src.data_prepareation - INFO - All prices are valid
2025-11-24 21:54:01,823 - src.data_prepareation - INFO - Customer lifetime values are consistent
2025-11-24 21:54:01,831 - src.data_prepareation - INFO - Created product_full: product_name + product_variant
2025-11-24 21:54:01,855 - src.data_prepareation - INFO - Created order_datetime from order_date and order_time
2025-11-24 21:54:01,861 - src.data_prepareation - INFO - Created temporal features: hour, day_of_week, month, year
2025-11-24 21:54:01,862 - src.data_

In [ ]:
aug = OrderAwareAugmentation(clean_df)
augmented_df = aug.run_augmentation(target_multiplier=1.5)

In [5]:
from src.abc_Data_augumentation import AugmentationFactory
pipeline = AugmentationFactory.create_standard_pipeline()
aug_df = pipeline.run_augmentation(clean_df)

2025-11-24 21:54:18,938 - src.abc_Data_augumentation - INFO - Created standard augmentation pipeline
2025-11-24 21:54:18,939 - src.abc_Data_augumentation - INFO - ================================================================================
2025-11-24 21:54:18,939 - src.abc_Data_augumentation - INFO - STEP 4: DATA AUGMENTATION WITH STRATEGY PATTERN
2025-11-24 21:54:18,939 - src.abc_Data_augumentation - INFO - ================================================================================
2025-11-24 21:54:18,939 - src.abc_Data_augumentation - INFO - Original size: 70,514
2025-11-24 21:54:18,939 - src.abc_Data_augumentation - INFO - Target size: 105,771
2025-11-24 21:54:18,939 - src.abc_Data_augumentation - INFO - Active strategies: ['temporal_jitter', 'variant_substitution', 'sequence_shuffle']
2025-11-24 21:54:18,941 - TemporalJitterStrategy - INFO - Applying temporal_jitter (jitter=2h, prob=0.3)
2025-11-24 21:54:18,975 - TemporalJitterStrategy - INFO - Generated 21154 augmented re

In [7]:
fe = OrderLevelFeatureEngineering(aug_df)
arrays, metadata = fe.run_feature_engineering(sequence_length=5, prediction_mode='basket')

2025-11-24 21:54:38,091 - src.data_feature_engineering - INFO - Order-Level Feature Engineering initialized
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO - ================================================================================
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO - STEP 3: ORDER-LEVEL FEATURE ENGINEERING
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO - ================================================================================
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO - Prediction mode: basket
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO -   'item' = Predict next items one-by-one
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO -   'basket' = Predict entire next order basket
2025-11-24 21:54:38,093 - src.data_feature_engineering - INFO - ================================================================================
2025-11-24 21:54:38,093 - src.data_feature_enginee

In [9]:
config = {
    'embedding_dim': 128,
    'hidden_dim': 128,
    'n_layers': 2,
    'dropout': 0.3,
    'epochs': 30,
    'patience': 10,
    'batch_size': 256,
    'learning_rate': 0.001
}
